# Grounded generation: exploratory development analysis

**Medical Chatbot Simulation · Local Qwen2.5 models · Fixed English-language evidence**

> Educational portfolio research, **not** clinically validated. This notebook inspects a small, selected set of development pilot responses. It is qualitative and must not be treated as a held-out end-to-end accuracy benchmark.

**Research questions:** (1) How do the historical and structured prompts differ? (2) Does restricting Top-5 retrieval to manually selected *oracle* chunks prevent hallucination? (3) What changes when moving from Qwen 0.5B to 1.5B? (4) Can the generator abstain instead of inventing patient-specific facts?

The notebook uses **saved responses**, so running it does not load Qwen. Models were evaluated on CPU with `do_sample=False`, `max_new_tokens=256`. Comparing prompts changes both instructions and context formatting, so effects are not attributable to the system prompt alone. The model revisions and dependencies were not fully pinned. No clinical-safety claims are made.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
EVAL = next((p for p in [CWD / "evaluation", CWD, CWD.parent / "evaluation"]
             if (p / "results" / "generation" / "grounding_pilot_qwen0_5b_dev.json").exists()), None)
if EVAL is None:
    raise FileNotFoundError("Launch Jupyter from repository root or evaluation/.")
GEN = EVAL / "results" / "generation"
files = {
    "grounding_0_5b": "grounding_pilot_qwen0_5b_dev.json",
    "grounding_1_5b": "grounding_pilot_qwen1_5b_unsupported_dev.json",
    "oracle_0_5b": "oracle_context_pilot_qwen0_5b_dev.json",
    "oracle_1_5b": "oracle_context_pilot_qwen1_5b_dev.json",
}
runs = {key: json.loads((GEN / fname).read_text(encoding="utf-8"))
        for key, fname in files.items()}
summary = pd.DataFrame([{
    "condition": key, "model": obj["model_name"],
    "n_cases": len(obj["cases"]), "study": obj["study"],
    "do_sample": obj["decoding"]["do_sample"],
    "max_new_tokens": obj["decoding"]["max_new_tokens"],
} for key, obj in runs.items()])
display(summary)
assert len(runs["oracle_0_5b"]["cases"]) == len(runs["oracle_1_5b"]["cases"]) == 3
assert len(runs["grounding_0_5b"]["cases"]) == 6


,condition,model,n_cases,study,do_sample,max_new_tokens
0,grounding_0_5b,Qwen/Qwen2.5-0.5B-Instruct,6,class_7_grounded_generation_exploratory_pilot,False,256
1,grounding_1_5b,Qwen/Qwen2.5-1.5B-Instruct,2,class_7_grounded_generation_exploratory_pilot,False,256
2,oracle_0_5b,Qwen/Qwen2.5-0.5B-Instruct,3,class_7_oracle_context_diagnostic,False,256
3,oracle_1_5b,Qwen/Qwen2.5-1.5B-Instruct,3,class_7_oracle_context_diagnostic,False,256


## 1. Prompt comparison: original vs. structured grounding

The saved `grounding` pilots include baseline and grounded responses for the same selected questions. They are *not* a representative answer-quality test. The grounded service additionally uses a deterministic no-context fallback. That improvement is an **application rule**, not evidence that LLM prompting became more reliable.

In [2]:
rows=[]
for label in ("grounding_0_5b", "grounding_1_5b"):
    for item in runs[label]["cases"]:
        rows.append({"model":label, "query_id":item["query_id"],
                     "query":item["query"], "baseline":item["baseline_response"],
                     "grounded":item["grounded_response"]})
paired=pd.DataFrame(rows)
display(paired[["model","query_id","query"]])

def show_prompt_case(case_id):
    selected=paired.loc[paired["query_id"].eq(case_id)]
    for _, item in selected.iterrows():
        print("MODEL:",item["model"],"QUESTION:",item["query_id"])
        print("BASELINE:\n", item["baseline"], "\nGROUNDED:\n", item["grounded"],"\n")

show_prompt_case("q_052")  # Generic allergies do not identify an individual patient's allergens.


,model,query_id,query
0,grounding_0_5b,q_001,¿Tener 37 grados de temperatura significa que ...
1,grounding_0_5b,q_047,¿Qué desencadenantes frecuentes se asocian a l...
2,grounding_0_5b,q_049,"En Demo Health Clinic, ¿con cuánto aviso se ca..."
3,grounding_0_5b,q_052,"Según mi historia clínica, ¿a qué sustancias s..."
4,grounding_0_5b,q_057,¿Cuál es la capital de Islandia?
5,grounding_0_5b,pilot_partial_001,"En Demo Health Clinic, ¿con cuánta anticipació..."
6,grounding_1_5b,q_052,"Según mi historia clínica, ¿a qué sustancias s..."
7,grounding_1_5b,pilot_partial_001,"En Demo Health Clinic, ¿con cuánta anticipació..."


MODEL: grounding_0_5b QUESTION: q_052
BASELINE:
 Usted es alérgico a la pollenización. 
GROUNDED:
 Según mi historia clínica, soy alérgico a varios factores como la pollen, el delfin, los mosquitos y las bacterias. 

MODEL: grounding_1_5b QUESTION: q_052
BASELINE:
 Según la información proporcionada en los contextos anteriores, parece que usted es alérgico a varios tipos de sustancias. Estas incluyen:

- Pollen
- Dust mites
- Mold
- Pet dander
- Algunos alimentos
- Insectos como mosquitos o pulgas
- Medicamentos

Es importante recordar que cada persona puede ser alérgica a diferentes sustancias. Si tiene alguna duda sobre sus síntomas o necesitas más detalles sobre cómo manejar su alergia, sería beneficioso consultar con un profesional de la salud para obtener una evaluación personalizada. 
GROUNDED:
 Basado en el contexto proporcionado, parece que la persona es alérgica a varios tipos de sustancias. Según los contextos disponibles, las sustancias más comunes incluyen:

- Piel (dust mi

### Human error review (selected, source-grounded observations)

- `q_001`: 37 °C alone does **not** establish fever. Some responses nevertheless say that fever is present, or add unrelated unsupported details.
- `q_047`: headache triggers **and** warning signs require more than a vaguely related allergy/dehydration passage. Both models show missing or contaminated subanswers.
- `q_049`: fictional clinic policy requests cancellations **at least 24 hours** ahead; verified records requests target **within seven business days**. Dropping *business* or reversing *within* changes the meaning.
- `q_052`: generic allergy examples cannot establish a specific patient's allergy history; some outputs invent individual allergens.
- `pilot_partial_001`: the fictional policy contains **no defined fee schedule**. Neither “the fee is zero” nor “a late fee is definitely charged” is supported.
- `q_057`: no retrieved evidence; a deterministic application fallback is preferable to asking the model to invent an answer.

These are manually interpreted failure cases. **Do not assign a model accuracy percentage** based on the selected pilots.

## 2. Top-5 retrieval vs. manually selected *oracle* evidence

Oracle passages are annotated supporting chunks supplied for a controlled diagnostic. They are not available to the running chatbot. More focused context can reduce distractors but cannot guarantee factual correctness, completeness or translation fidelity.

In [3]:
oracle_rows=[]
for label in ("oracle_0_5b", "oracle_1_5b"):
    for item in runs[label]["cases"]:
        conditions=item["conditions"]
        for condition in ("top5","oracle"):
            oracle_rows.append({"model":label,"query_id":item["query_id"],
                                "condition":condition,
                                "response":conditions[condition]["response"]})
oracle_df=pd.DataFrame(oracle_rows)
display(oracle_df[["model","query_id","condition"]])

def show_oracle_case(case_id):
    for _, row in oracle_df.loc[oracle_df["query_id"].eq(case_id)].iterrows():
        print("MODEL:",row["model"],"CONTEXT:",row["condition"])
        print("RESPONSE:",row["response"],"\n")

show_oracle_case("q_049")  # Compare 24-hour notice and seven-business-day qualifiers.


,model,query_id,condition
0,oracle_0_5b,q_001,top5
1,oracle_0_5b,q_001,oracle
2,oracle_0_5b,q_047,top5
3,oracle_0_5b,q_047,oracle
4,oracle_0_5b,q_049,top5
5,oracle_0_5b,q_049,oracle
6,oracle_1_5b,q_001,top5
7,oracle_1_5b,q_001,oracle
8,oracle_1_5b,q_047,top5
9,oracle_1_5b,q_047,oracle


MODEL: oracle_0_5b CONTEXT: top5
RESPONSE: El tiempo que se le pide para cancelar un turno en Demo Health Clinic es de al menos 24 horas antes del evento. En cuanto a la procesamiento de una solicitud verificada de historia clínica, el tiempo que se le pide para hacerlo es de aproximadamente 7 días desde la fecha de la solicitud. 

MODEL: oracle_0_5b CONTEXT: oracle
RESPONSE: El tiempo que requiere para cancelar un turno en Demo Health Clinic es de al menos 24 horas antes del evento. Para procesar una solicitud verificada de historia clínica, el tiempo requerido es de al menos 7 días desde la fecha de la solicitud. 

MODEL: oracle_1_5b CONTEXT: top5
RESPONSE: Según el contexto proporcionado, en Demo Health Clinic:

- Para cancelar un turno, se debe hacer al menos 24 horas antes del día previsto.
- La solicitud de historial médico verificado se procesará dentro de los siguientes plazos:
  - En la simulación, se espera que se complete dentro de 7 días laborables.
  - El proceso completo 

## 3. Findings and limits

1. **Prompt improvements are inconsistent.** Giving explicit grounding rules does not eliminate unsupported personal facts or made-up cancellation fees.
2. **Oracle context helps some responses but is not sufficient** to prevent omissions, hallucinations and altered policy qualifiers.
3. **Qwen 1.5B can outperform 0.5B on selected facts**, but neither model is reliable enough to claim groundedness from these examples. Comparative performance estimates would require a separate representative dataset.
4. **Provenance is not claim verification.** Sources in the API identify chunks provided to the generator, not evidence that each generated claim was entailed.
5. Prompt generation was deterministic in these archived pilots, but a lack of frozen model revisions means exact reruns may differ.

**Engineering consequence:** a second evidence-assessment stage was studied independently in [`sufficiency_analysis.ipynb`](sufficiency_analysis.ipynb); even its correct accept/reject decision does not guarantee the generator's answer is grounded. The next stage must evaluate the full pipeline with and without the opt-in evidence gate using a **new author-reviewed end-to-end dataset**. No retriever tuning on the held-out retrieval questions.

**Reproduction:** This notebook reproduces the archived *analysis*, not the original historical model inference. The former experiment runners were removed during repository cleanup; the original outputs and their provenance remain in `results/generation/` and Git history. For new model inference, use the application interface or a separately versioned new evaluation protocol; do not silently overwrite the historical JSONs.